# 12 — Faster Batch Pipeline v2

This version keeps the existing end-to-end pipeline but reduces the expensive
fact-check workload.

Main changes:

- at most **4** high-priority claims;
- up to **4 claims in one Ollama verification request**;
- **2 evidence sources** per claim with shorter excerpts;
- at most **2 rewrite attempts**;
- no embedded video preview, so saving or committing the notebook does not
  place the video inside the `.ipynb` file.

Required supporting files:

```text
educational_shorts/batch_fast_v2.py
educational_shorts/fact_checker_fast_v2.py
```

## Load the project

In [1]:
import sys
import time
from datetime import datetime, timezone
from pathlib import Path


def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "educational_shorts").is_dir():
            return candidate

    raise FileNotFoundError(
        "Could not find the project root containing educational_shorts/."
    )


PROJECT_ROOT = find_project_root(Path.cwd())

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from educational_shorts.batch import TopicBatchPlan
from educational_shorts.batch_fast_v2 import (
    FastBatchConfig,
    preflight_batch_pipeline,
    run_batch_pipeline_fast,
    summarize_batch_manifest,
)
from educational_shorts.schemas import VideoTopicList
from educational_shorts.topic_library import (
    claim_next_topic,
    mark_topic_completed,
    mark_topic_failed,
    reset_failed_topic,
)

print("NOTEBOOK_12_BATCH_PIPELINE_FAST_V2")
print(f"Project root: {PROJECT_ROOT}")

NOTEBOOK_12_BATCH_PIPELINE_FAST_V2
Project root: c:\Users\hitch\python_files\educational_shorts


## Configuration

In [2]:
ROOT_CATEGORY = "Science"

# Use None to choose from every approved topic in the database.
TOPIC_CATEGORY_ROOT = ROOT_CATEGORY
TOPIC_DATABASE_PATH = PROJECT_ROOT / "data" / "topic_library.db"

PAUSE_ON_MANUAL_REVIEW = False
CONTINUE_ON_ERROR = False
SKIP_EXISTING_FINAL = True

GAMEPLAY_SUBDIRECTORY = "subway_surfers"
BACKGROUND_VIDEO_FILENAME = None

TTS_VOICE = "am_michael"
TTS_SPEED = 1.0

CAPTION_STYLE = "phrase"
ASS_FONT_NAME = "Arial"
ASS_FONT_SIZE = 72
ASS_MARGIN_V = 300

OUTPUT_WIDTH = 1080
OUTPUT_HEIGHT = 1920
CROP_ANCHOR_Y = "top"
LOOP_BACKGROUND = True

# High-impact fact-check speed controls.
FACT_CHECK_MAX_CLAIMS = 4
FACT_CHECK_VERIFICATION_BATCH_SIZE = 4
FACT_CHECK_MAX_SEARCH_RESULTS = 5
FACT_CHECK_MAX_SOURCES_PER_CLAIM = 2
FACT_CHECK_MAX_EXCERPT_CHARS = 700
FACT_CHECK_CACHE_TTL_DAYS = 90
FACT_CHECK_REWRITE_MAX_ATTEMPTS = 2

config = FastBatchConfig(
    root_category=ROOT_CATEGORY,
    pause_on_manual_review=PAUSE_ON_MANUAL_REVIEW,
    gameplay_subdirectory=GAMEPLAY_SUBDIRECTORY,
    background_video_filename=BACKGROUND_VIDEO_FILENAME,
    tts_voice=TTS_VOICE,
    tts_speed=TTS_SPEED,
    caption_style=CAPTION_STYLE,
    ass_font_name=ASS_FONT_NAME,
    ass_font_size=ASS_FONT_SIZE,
    ass_margin_v=ASS_MARGIN_V,
    output_width=OUTPUT_WIDTH,
    output_height=OUTPUT_HEIGHT,
    crop_anchor_y=CROP_ANCHOR_Y,
    loop_background=LOOP_BACKGROUND,
    skip_existing_final=SKIP_EXISTING_FINAL,
    continue_on_error=CONTINUE_ON_ERROR,
    fact_check_max_claims=FACT_CHECK_MAX_CLAIMS,
    fact_check_verification_batch_size=(
        FACT_CHECK_VERIFICATION_BATCH_SIZE
    ),
    fact_check_max_search_results=FACT_CHECK_MAX_SEARCH_RESULTS,
    fact_check_max_sources_per_claim=(
        FACT_CHECK_MAX_SOURCES_PER_CLAIM
    ),
    fact_check_max_excerpt_chars=FACT_CHECK_MAX_EXCERPT_CHARS,
    fact_check_cache_ttl_days=FACT_CHECK_CACHE_TTL_DAYS,
    fact_check_rewrite_max_attempts=(
        FACT_CHECK_REWRITE_MAX_ATTEMPTS
    ),
)

print(config.model_dump_json(indent=2))

{
  "root_category": "Science",
  "tree_filename": null,
  "category_path_override": null,
  "topic_candidate_count": 10,
  "min_category_depth": 2,
  "max_category_depth": 3,
  "category_selection_seed": 42,
  "topic_generation_seed": 42,
  "topic_generation_temperature": 0.7,
  "target_seconds": 60,
  "section_count": 4,
  "outline_temperature": 0.4,
  "outline_seed": 42,
  "target_words_per_minute": 145,
  "script_temperature": 0.5,
  "script_seed": 42,
  "editor_minimum_seconds": 40,
  "editor_maximum_seconds": 60,
  "editor_temperature": 0.4,
  "editor_seed": 42,
  "fact_check_minimum_words": 105,
  "fact_check_maximum_words": 150,
  "fact_check_temperature": 0.1,
  "fact_check_seed": 42,
  "metadata_temperature": 0.5,
  "metadata_seed": 42,
  "metadata_max_attempts": 4,
  "pause_on_manual_review": false,
  "tts_language_code": "a",
  "tts_voice": "am_michael",
  "tts_speed": 1.0,
  "tts_chunk_pause_ms": 80,
  "tts_segment_pause_ms": 240,
  "tts_target_peak_dbfs": -1.0,
  "normali

## Preflight check

In [3]:
preflight = preflight_batch_pipeline(
    project_root=PROJECT_ROOT,
    config=config,
)

for name, value in preflight.items():
    print(f"{name}: {value}")

project_root: c:\Users\hitch\python_files\educational_shorts
tree_path: c:\Users\hitch\python_files\educational_shorts\data\knowledge_tree\science.json
tree_root: Science
gameplay_directory: c:\Users\hitch\python_files\educational_shorts\data\gameplay\subway_surfers
background_video: c:\Users\hitch\python_files\educational_shorts\data\gameplay\subway_surfers\ScreenRecording_07-23-2026 10-14-28_1.mp4
ffmpeg: C:\Users\hitch\AppData\Local\Microsoft\WinGet\Packages\Gyan.FFmpeg_Microsoft.Winget.Source_8wekyb3d8bbwe\ffmpeg-8.1.2-full_build\bin\ffmpeg.EXE
ffprobe: C:\Users\hitch\AppData\Local\Microsoft\WinGet\Packages\Gyan.FFmpeg_Microsoft.Winget.Source_8wekyb3d8bbwe\ffmpeg-8.1.2-full_build\bin\ffprobe.EXE


## Claim the next approved topic

In [4]:
claimed_topic = claim_next_topic(
    database_path=TOPIC_DATABASE_PATH,
    category_root=TOPIC_CATEGORY_ROOT,
)

if claimed_topic is None:
    raise RuntimeError(
        "No approved topics are available in the topic library. "
        "Run Notebook 03 to add more approved topics."
    )

topic_id, selected_topic, category_path = claimed_topic
now = datetime.now(timezone.utc)

topic_plan = TopicBatchPlan(
    run_id=now.strftime("%Y%m%dT%H%M%S%fZ"),
    category_path=category_path,
    topics_file=str(TOPIC_DATABASE_PATH),
    created_at_utc=now.isoformat(),
    topics=VideoTopicList(topics=[selected_topic]),
)

print(f"Database topic ID: {topic_id}")
print(f"Topic: {selected_topic.title}")
print(f"Category: {' > '.join(category_path)}")
print(f"Learning objective: {selected_topic.learning_objective}")
print("Database status: processing")

Database topic ID: 2
Topic: How Do Bacteria Communicate?
Category: Science > Biology > Microbiology
Learning objective: Understand how bacteria use chemical signals to communicate and coordinate behavior.
Database status: processing


## Run the pipeline

This is the long-running cell. It prints the number of claims and verification
batches so the fact-check workload is visible.

In [5]:
started = time.perf_counter()

try:
    batch_manifest = run_batch_pipeline_fast(
        project_root=PROJECT_ROOT,
        plan=topic_plan,
        selected_topic_indexes=[0],
        config=config,
    )

    item = batch_manifest.items[0]

    if item.status in {"completed", "skipped_existing"}:
        final_video = item.paths.get("final_video")

        if not final_video:
            raise RuntimeError(
                "The pipeline reported completion but did not provide "
                "a final video path."
            )

        mark_topic_completed(
            database_path=TOPIC_DATABASE_PATH,
            topic_id=topic_id,
            final_video_path=Path(final_video),
        )
        print("Topic-library status: completed")

    elif item.status == "manual_review":
        # The current topic-library schema has no manual_review status.
        # Return the topic to the approved queue rather than marking it failed.
        reset_failed_topic(
            database_path=TOPIC_DATABASE_PATH,
            topic_id=topic_id,
        )
        print("Topic-library status: requeued for manual review")

    else:
        mark_topic_failed(
            database_path=TOPIC_DATABASE_PATH,
            topic_id=topic_id,
            message=f"{item.status}: {item.message}",
        )
        print("Topic-library status: failed")

except Exception as error:
    mark_topic_failed(
        database_path=TOPIC_DATABASE_PATH,
        topic_id=topic_id,
        message=f"{type(error).__name__}: {error}",
    )
    raise

finally:
    elapsed_minutes = (time.perf_counter() - started) / 60
    print(f"Total pipeline time: {elapsed_minutes:.1f} minutes")

Fast fact check enabled: up to 4 claims, 4 claims per verification request, 2 sources per claim.

ITEM 1/1: How Do Bacteria Communicate?
Retrieving 1/4: Bacteria use chemical signals called quorum sensing to determine their population density.
  2 source(s) from web
Retrieving 2/4: Bacteria release signaling molecules into their environment, which are detected by other bacteria to trigger responses.
  2 source(s) from web
Retrieving 3/4: Bacterial communication leads to the formation of protective biofilms or a shift from harmless to harmful behavior.
  2 source(s) from web
Retrieving 4/4: Scientific research on bacterial communication is aimed at developing strategies to combat infections and create novel treatments.
  2 source(s) from web
Verifying batch 1/1: 4 claim(s) [claim_1, claim_2, claim_3, claim_4]
Status: failed
Final stage: fact_checking
ValueError: Could not produce a valid corrected script after 2 attempts. attempt 1: 89 words; attempt 2: 91 words
Topic-library status: fa

## Review the result

In [6]:
for name, value in summarize_batch_manifest(
    batch_manifest
).items():
    print(f"{name}: {value}")

item = batch_manifest.items[0]

print()
print(f"Status: {item.status}")
print(f"Final stage: {item.final_stage}")
print(f"Message: {item.message}")
print(f"Fact-check verdict: {item.fact_check_verdict}")
print(f"Requires manual review: {item.requires_manual_review}")

print()
print("Completed stages:")
for stage in item.stages_completed:
    print(f"  - {stage}")

print()
print("Saved paths:")
for name, path in item.paths.items():
    print(f"  {name}: {path}")

run_id: 20260724T165230566409Z
status: aborted
selected_topics: 1
completed: 0
manual_review: 0
skipped_existing: 0
failed: 1
output_directory: c:\Users\hitch\python_files\educational_shorts\data\batch_runs\20260724T165230566409Z
manifest: c:\Users\hitch\python_files\educational_shorts\data\batch_runs\20260724T165230566409Z\batch_manifest.json

Status: failed
Final stage: fact_checking
Message: ValueError: Could not produce a valid corrected script after 2 attempts. attempt 1: 89 words; attempt 2: 91 words
Fact-check verdict: None
Requires manual review: None

Completed stages:
  - outline_generation
  - script_generation
  - script_editing

Saved paths:
  outline: c:\Users\hitch\python_files\educational_shorts\data\outlines\how_do_bacteria_communicate.json
  script: c:\Users\hitch\python_files\educational_shorts\data\scripts\how_do_bacteria_communicate.json
  edited_script: c:\Users\hitch\python_files\educational_shorts\data\edited_scripts\how_do_bacteria_communicate.json


## Final video path

This notebook deliberately does not embed the video. Open the printed path in
your media player or file explorer.

In [7]:
final_video_path = item.paths.get("final_video")

if final_video_path:
    print(final_video_path)
else:
    print("No final video was produced.")

No final video was produced.
